# Single-Property Direction Validation (Span Sweep) - cLogP

This notebook is the corrected **single-property** prototype for Step 6, retargeted to **cLogP** with a **span sweep** over traversal scales.

It is designed to validate one chemical property at a time while staying aligned with:

- the **Step 4 v2** saved linear directions,
- the corrected `final_nat_linearpool.py` decoder,
- the project goal of testing **raw vs residual vs confound vs random** directions by decoded traversal behavior.

Main corrections compared with the earlier compact single-property notebook:

- use the **corrected model helper** directly for free-length decoding,
- keep traversal geometry in **standardized latent space** and only map back to raw latent at decode time,
- use **candidate packs** with orthogonal latent noise instead of a single deterministic decode,
- build a **representative path** with dynamic programming instead of taking one brittle decode per step,
- keep **free-length** as the main mode, with **fixed-length** as an optional diagnostic control,
- recompute the target property, confounds, and residualized target on the decoded molecules,
- log collapse-sensitive diagnostics such as validity, uniqueness, path constancy, continuity, and predicted-length drift.
- repeat each direction across multiple latent traversal spans to test whether decoder movement appears only after step-size escalation.

## 1. Imports and notebook configuration

In [ ]:
import hashlib
import importlib.util
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import selfies as sf
import torch
import torch.nn.functional as F
from IPython.display import display
from rdkit import Chem, DataStructs
from rdkit.Chem import Crippen, Descriptors, Draw, QED, rdFingerprintGenerator, rdMolDescriptors
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROPERTY = "HBA"
PROPERTY_RESID_NAME = f"resid_{PROPERTY}"

USE_STEP4_ARTIFACTS = True
ALLOW_INLINE_FALLBACK = True

SMOKE_MODE = False
SEED_COUNT = 4 if SMOKE_MODE else 12
PACK_SIZE = 4 if SMOKE_MODE else 16
STEP_GRID = np.arange(-2, 3, dtype=int) if SMOKE_MODE else np.arange(-4, 5, dtype=int)
DECODE_MODES = ["free_length", "fixed_length_seed_pred"]
BENCHMARK_PAIR_COUNT = 64 if SMOKE_MODE else 200

RIDGE_ALPHAS = np.logspace(-4, 4, 17)
TARGET_STEP_FRACTION_OF_STD = 0.25
DEFAULT_TOTAL_SPAN_FACTOR_OF_BENCHMARK = 0.50
MAX_TOTAL_SPAN_FACTOR_OF_BENCHMARK = 0.75
SPAN_SWEEP_MULTIPLIERS = [1.0, 4.0, 16.0, 64.0]
SPAN_SWEEP_MAX_FACTOR_OF_BENCHMARK = 2.0

SIGMA_STD_LOW = 0.10 if SMOKE_MODE else 0.05
SIGMA_STD_HIGH = 0.20 if SMOKE_MODE else 0.10
EARLY_UNIQUENESS_THRESHOLD = 0.75

C_COLS = ["selfies_len_tokens", "branch_token_count", "ring_token_count", "token_entropy"]
Y_COLS = [
    "MolWt",
    "ExactMolWt",
    "HeavyAtomCount",
    "cLogP",
    "TPSA",
    "HBD",
    "HBA",
    "NumRotatableBonds",
    "RingCount",
    "AromaticRingCount",
    "FractionCSP3",
    "NumSpiroAtoms",
    "NumBridgeheadAtoms",
    "BertzCT",
    "QED",
]
SEED_BANDS = ["low", "mid", "high"]
REPRESENTATIVE_VARIANTS_TO_STRIP = ["raw", "residual"]
STRIP_DECODE_MODE = "free_length"
STRIP_SPAN_MULTIPLIER = SPAN_SWEEP_MULTIPLIERS[-1]

assert PROPERTY in Y_COLS, f"PROPERTY={PROPERTY!r} must be one of the supported RDKit properties."

## 2. Resolve project paths and output folders

In [ ]:
def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for base in [start] + list(start.parents):
        if (base / "smiles_selfies_full.csv").exists():
            return base
        if (base / "transformer_vae").exists() and (base.parent / "smiles_selfies_full.csv").exists():
            return base.parent
    raise FileNotFoundError(
        "Could not locate the project root. Run the notebook from the repo root or a subdirectory that can reach "
        "`smiles_selfies_full.csv`."
    )

def first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None

PROJECT_ROOT = find_project_root()
TRANSFORMER_DIR = PROJECT_ROOT / "transformer_vae"
DATA_CSV = PROJECT_ROOT / "smiles_selfies_full.csv"

MODEL_PY = first_existing([
    TRANSFORMER_DIR / "models" / "final_nat_linearpool.py",
    PROJECT_ROOT / "final_nat_linearpool.py",
    Path("/mnt/data/final_nat_linearpool.py"),
])

CKPT_PATH = first_existing([
    TRANSFORMER_DIR / "trained_models" / "final(H256-L512-linpool).pt",
    PROJECT_ROOT / "trained_models" / "final(H256-L512-linpool).pt",
])

STEP3_DIR = PROJECT_ROOT / "artifacts" / "confounds_step3"
STEP4_DIR = PROJECT_ROOT / "artifacts" / "directions_step4_v2"

OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "single_property_direction_validation_span_sweep" / PROPERTY
TABLES_DIR = OUTPUT_DIR / "tables"
PLOTS_DIR = OUTPUT_DIR / "plots"
STRIPS_DIR = PLOTS_DIR / "representative_strips"

for folder in (OUTPUT_DIR, TABLES_DIR, PLOTS_DIR, STRIPS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

assert DATA_CSV.exists(), f"Missing dataset: {DATA_CSV}"
assert MODEL_PY is not None and MODEL_PY.exists(), "Could not resolve final_nat_linearpool.py"
assert CKPT_PATH is not None and CKPT_PATH.exists(), "Could not resolve the VAE checkpoint"
assert STEP3_DIR.exists(), f"Missing Step 3 artifacts: {STEP3_DIR}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
print("project_root:", PROJECT_ROOT)
print("model_py:", MODEL_PY)
print("ckpt_path:", CKPT_PATH)
print("step3_dir:", STEP3_DIR)
print("step4_dir:", STEP4_DIR if STEP4_DIR.exists() else "(missing; inline fallback will be used)")
print("output_dir:", OUTPUT_DIR)

## 3. Load dataset, Step 3 panel, tokenization, latent cache, and corrected model

In [ ]:
def full_molecule_tokens_to_ids(tokens, tok2id, sos_id, eos_id):
    return np.array([sos_id] + [tok2id[t] for t in tokens] + [eos_id], dtype=np.int64)

def decode_tokens_to_selfies(token_ids, id2tok, pad_id=0, sos_id=1, eos_id=2):
    if isinstance(token_ids, torch.Tensor):
        token_ids = token_ids.detach().cpu().tolist()
    tokens = []
    for t in token_ids:
        t = int(t)
        if t == eos_id:
            break
        if t in (pad_id, sos_id):
            continue
        tok = id2tok.get(t)
        if tok is None:
            continue
        if tok in {"<PAD>", "<SOS>", "<EOS>"}:
            continue
        tokens.append(tok)
    return "".join(tokens)

def selfies_to_smiles(selfies_str):
    if selfies_str is None:
        return None
    if not isinstance(selfies_str, str):
        selfies_str = str(selfies_str)
    if selfies_str == "":
        return None
    try:
        return sf.decoder(selfies_str)
    except Exception:
        return None

def smiles_to_mol(smiles_str):
    if smiles_str is None:
        return None
    if not isinstance(smiles_str, str):
        smiles_str = str(smiles_str)
    if smiles_str == "":
        return None
    try:
        return Chem.MolFromSmiles(smiles_str)
    except Exception:
        return None

def canon_smiles(smiles_str):
    mol = smiles_to_mol(smiles_str)
    if mol is None:
        return None
    try:
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None

df = pd.read_csv(DATA_CSV)
panel = pd.read_csv(STEP3_DIR / "panels_step3_with_residuals.csv")

assert len(df) == len(panel), "Dataset and Step 3 panel row counts differ."
if {"smiles", "selfies"}.issubset(df.columns) and {"smiles", "selfies"}.issubset(panel.columns):
    assert df["smiles"].astype(str).equals(panel["smiles"].astype(str)), "Mismatch in smiles between dataset and Step 3 panel."
    assert df["selfies"].astype(str).equals(panel["selfies"].astype(str)), "Mismatch in selfies between dataset and Step 3 panel."

df["tokens"] = df["selfies"].apply(lambda x: list(sf.split_selfies(x)))
all_tokens = [tok for seq in df["tokens"] for tok in seq]

PAD = "<PAD>"
SOS = "<SOS>"
EOS = "<EOS>"
vocab = [PAD, SOS, EOS] + sorted(set(all_tokens))

tok2id = {tok: idx for idx, tok in enumerate(vocab)}
id2tok = {idx: tok for tok, idx in tok2id.items()}

PAD_ID = tok2id[PAD]
SOS_ID = tok2id[SOS]
EOS_ID = tok2id[EOS]

df["token_ids"] = df["tokens"].apply(lambda toks: full_molecule_tokens_to_ids(toks, tok2id, SOS_ID, EOS_ID))
max_len = int(df["token_ids"].apply(len).max())

padded_data = np.zeros((len(df), max_len), dtype=np.int64)
for i, seq in enumerate(df["token_ids"].tolist()):
    padded_data[i, : len(seq)] = seq

split = panel["split"].astype(str).str.strip().str.lower()
train_mask = split.eq("train").to_numpy()
val_mask = split.eq("val").to_numpy()
test_mask = split.eq("test").to_numpy()

assert train_mask.sum() > 0 and val_mask.sum() > 0 and test_mask.sum() > 0

train_idx = np.flatnonzero(train_mask).astype(int)
val_idx = np.flatnonzero(val_mask).astype(int)
test_idx = np.flatnonzero(test_mask).astype(int)

z_raw = np.load(STEP3_DIR / "Z_mu.npy").astype(np.float32)
assert z_raw.shape[0] == len(panel), "Latent cache row count mismatch."

z_mean_train = z_raw[train_mask].mean(axis=0).astype(np.float32)
z_std_train = z_raw[train_mask].std(axis=0).astype(np.float32)
z_std_safe = np.where(np.abs(z_std_train) < 1e-8, 1.0, z_std_train).astype(np.float32)
z_std_all = ((z_raw - z_mean_train) / z_std_safe).astype(np.float32)

val_panel = panel.iloc[val_idx].reset_index(drop=True).copy()
val_panel["val_pos"] = np.arange(len(val_panel), dtype=int)
val_panel["dataset_index"] = val_idx.astype(int)
val_panel["canonical_smiles"] = val_panel["smiles"].apply(canon_smiles)

val_data = padded_data[val_mask]

spec = importlib.util.spec_from_file_location("final_nat_linearpool", str(MODEL_PY))
final_nat_linearpool = importlib.util.module_from_spec(spec)
spec.loader.exec_module(final_nat_linearpool)

def load_final_model_safe(vocab_size, max_len, ckpt_path, device):
    model = final_nat_linearpool.VaeTransformer(
        vocab_size=vocab_size,
        hidden_size=final_nat_linearpool.FINAL_HIDDEN_SIZE,
        latent_size=final_nat_linearpool.FINAL_LATENT_SIZE,
        max_len=max_len,
        attn_heads=final_nat_linearpool.FINAL_ATTENTION_HEADS,
        num_slots=final_nat_linearpool.FINAL_NUM_SLOTS,
        layers=final_nat_linearpool.FINAL_LAYERS,
    ).to(device)
    load_kwargs = {"map_location": device}
    try:
        ckpt = torch.load(str(ckpt_path), weights_only=True, **load_kwargs)
    except TypeError:
        ckpt = torch.load(str(ckpt_path), **load_kwargs)
    state_dict = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    model.load_state_dict(state_dict)
    model.eval()
    return model

model = load_final_model_safe(vocab_size=len(vocab), max_len=max_len, ckpt_path=CKPT_PATH, device=device)

print("rows:", len(panel))
print("train / val / test:", int(train_mask.sum()), int(val_mask.sum()), int(test_mask.sum()))
print("latent_dim:", int(z_raw.shape[1]))
print("max_len:", max_len)

## 4. Load Step 4 directions when available, otherwise rebuild the single-property probe inline

In [ ]:
def stable_random_index(name, size):
    digest = hashlib.md5(str(name).encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % int(size)

def unit_vector(v):
    arr = np.asarray(v, dtype=np.float32)
    norm = float(np.linalg.norm(arr))
    if norm < 1e-12:
        return np.zeros_like(arr)
    return arr / norm

def cosine(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    na = float(np.linalg.norm(a))
    nb = float(np.linalg.norm(b))
    if na < 1e-12 or nb < 1e-12:
        return np.nan
    return float(np.dot(a, b) / (na * nb))

def coef_raw_to_std(w_raw):
    arr = np.asarray(w_raw, dtype=np.float32)
    return arr * z_std_safe

def raw_to_std_latents(z_values):
    arr = np.asarray(z_values, dtype=np.float32)
    return (arr - z_mean_train) / z_std_safe

def std_to_raw_latents(z_values_std):
    arr = np.asarray(z_values_std, dtype=np.float32)
    return z_mean_train + arr * z_std_safe

def safe_r2(y_true, y_pred, min_n=10):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if int(mask.sum()) < min_n:
        return np.nan
    if float(np.std(y_true[mask])) < 1e-12:
        return np.nan
    return float(r2_score(y_true[mask], y_pred[mask]))

def safe_mae(y_true, y_pred, min_n=10):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if int(mask.sum()) < min_n:
        return np.nan
    return float(mean_absolute_error(y_true[mask], y_pred[mask]))

def fit_probe_single(target_name, y_all):
    y_all = np.asarray(y_all, dtype=float)
    tr = train_mask & np.isfinite(y_all)
    va = val_mask & np.isfinite(y_all)
    te = test_mask & np.isfinite(y_all)

    if int(tr.sum()) == 0:
        raise ValueError(f"No finite training rows for {target_name}.")

    xtr = z_std_all[tr]
    xva = z_std_all[va]
    xte = z_std_all[te]
    ytr = y_all[tr]
    yva = y_all[va]
    yte = y_all[te]

    y_mu = float(np.nanmean(ytr))
    y_sd = float(np.nanstd(ytr))
    if not np.isfinite(y_sd) or y_sd < 1e-8:
        y_sd = 1.0
    ytr_std = (ytr - y_mu) / y_sd

    best_alpha = float(RIDGE_ALPHAS[0])
    best_r2 = -np.inf
    for alpha in RIDGE_ALPHAS:
        probe = Ridge(alpha=float(alpha), fit_intercept=True)
        probe.fit(xtr, ytr_std)
        pred_val = probe.predict(xva) * y_sd + y_mu
        r2_val = safe_r2(yva, pred_val)
        if np.isfinite(r2_val) and r2_val > best_r2:
            best_r2 = float(r2_val)
            best_alpha = float(alpha)

    probe = Ridge(alpha=best_alpha, fit_intercept=True)
    probe.fit(xtr, ytr_std)

    coef_std = np.asarray(probe.coef_, dtype=np.float32)
    intercept_std = float(probe.intercept_)

    pred_train = probe.predict(xtr) * y_sd + y_mu
    pred_val = probe.predict(xva) * y_sd + y_mu
    pred_test = probe.predict(xte) * y_sd + y_mu

    w_raw = ((y_sd * coef_std) / z_std_safe).astype(np.float32)
    b_raw = float(
        y_mu + y_sd * (
            intercept_std - float(np.dot((z_mean_train / z_std_safe).astype(np.float32), coef_std))
        )
    )
    w_std = coef_raw_to_std(w_raw)

    return {
        "target": target_name,
        "alpha": best_alpha,
        "target_mean_train": y_mu,
        "target_std_train": y_sd,
        "coef_std": coef_std,
        "intercept_std": intercept_std,
        "w_raw": w_raw,
        "b_raw": b_raw,
        "w_std": w_std,
        "direction_std_unit": unit_vector(w_std),
        "w_std_norm": float(np.linalg.norm(w_std)),
        "r2_train": safe_r2(ytr, pred_train),
        "r2_val": safe_r2(yva, pred_val),
        "r2_test": safe_r2(yte, pred_test),
        "mae_train": safe_mae(ytr, pred_train),
        "mae_val": safe_mae(yva, pred_val),
        "mae_test": safe_mae(yte, pred_test),
        "source": "inline_fit",
    }

def fit_property_residualizer(property_name):
    y_all = pd.to_numeric(panel[property_name], errors="coerce").to_numpy(dtype=float)
    C_all = panel[C_COLS].to_numpy(dtype=float)

    tr = train_mask & np.isfinite(y_all) & np.isfinite(C_all).all(axis=1)
    if int(tr.sum()) == 0:
        raise ValueError(f"No finite confound rows for residualizer of {property_name}.")

    c_scaler = StandardScaler()
    y_scaler = StandardScaler()

    C_train_s = c_scaler.fit_transform(C_all[tr])
    y_train_s = y_scaler.fit_transform(y_all[tr].reshape(-1, 1)).ravel()

    conf_model = RidgeCV(alphas=RIDGE_ALPHAS)
    conf_model.fit(C_train_s, y_train_s)

    pred_all = np.full(len(panel), np.nan, dtype=float)
    valid = np.isfinite(C_all).all(axis=1)
    if valid.any():
        pred_s = conf_model.predict(c_scaler.transform(C_all[valid]))
        pred_all[valid] = y_scaler.inverse_transform(np.asarray(pred_s).reshape(-1, 1)).ravel()

    resid_all = y_all - pred_all

    metrics = {}
    for split_name, mask in [("train", train_mask), ("val", val_mask), ("test", test_mask)]:
        m = mask & np.isfinite(y_all) & np.isfinite(pred_all)
        metrics[f"r2_{split_name}"] = safe_r2(y_all[m], pred_all[m])
        metrics[f"mae_{split_name}"] = safe_mae(y_all[m], pred_all[m])

    return {
        "property": property_name,
        "model": conf_model,
        "c_scaler": c_scaler,
        "y_scaler": y_scaler,
        "pred_all": pred_all,
        "resid_all": resid_all,
        **metrics,
    }

STEP4_AVAILABLE = bool(
    USE_STEP4_ARTIFACTS
    and STEP4_DIR.exists()
    and (STEP4_DIR / "directions_raw_v2.npz").exists()
    and (STEP4_DIR / "directions_resid_v2.npz").exists()
    and (STEP4_DIR / "r2_raw_v2.csv").exists()
    and (STEP4_DIR / "r2_resid_v2.csv").exists()
    and (STEP4_DIR / "cosine_vs_confounds_raw_v2.csv").exists()
)

if STEP4_AVAILABLE:
    directions_raw_npz = np.load(STEP4_DIR / "directions_raw_v2.npz", allow_pickle=True)
    directions_resid_npz = np.load(STEP4_DIR / "directions_resid_v2.npz", allow_pickle=True)
    r2_raw_df = pd.read_csv(STEP4_DIR / "r2_raw_v2.csv")
    r2_resid_df = pd.read_csv(STEP4_DIR / "r2_resid_v2.csv")
    cos_raw_df = pd.read_csv(STEP4_DIR / "cosine_vs_confounds_raw_v2.csv")
    raw_name_to_idx = {str(x): i for i, x in enumerate(directions_raw_npz["target_names"].tolist())}
    resid_name_to_idx = {str(x): i for i, x in enumerate(directions_resid_npz["target_names"].tolist())}
    step4_random_unit_dirs = None
    if (STEP4_DIR / "random_controls_v2.npz").exists():
        step4_random_unit_dirs = np.asarray(
            np.load(STEP4_DIR / "random_controls_v2.npz", allow_pickle=True)["random_unit_dirs"],
            dtype=np.float32,
        )
else:
    directions_raw_npz = None
    directions_resid_npz = None
    r2_raw_df = pd.DataFrame()
    r2_resid_df = pd.DataFrame()
    cos_raw_df = pd.DataFrame()
    raw_name_to_idx = {}
    resid_name_to_idx = {}
    step4_random_unit_dirs = None

CONFOUND_DIRECTION_FILES = {
    "selfies_len_tokens": STEP3_DIR / "confound_directions" / "w_length.npy",
    "branch_token_count": STEP3_DIR / "confound_directions" / "w_branch.npy",
    "ring_token_count": STEP3_DIR / "confound_directions" / "w_ring.npy",
    "token_entropy": STEP3_DIR / "confound_directions" / "w_entropy.npy",
}
saved_confound_raw_map = {
    name: np.asarray(np.load(path), dtype=np.float32)
    for name, path in CONFOUND_DIRECTION_FILES.items()
    if path.exists()
}

def load_saved_probe(target_name, residual=False):
    if residual:
        if target_name not in resid_name_to_idx:
            return None
        row = r2_resid_df[r2_resid_df["target"] == target_name]
        if row.empty:
            return None
        row = row.iloc[0]
        w_raw = np.asarray(directions_resid_npz["w"][resid_name_to_idx[target_name]], dtype=np.float32)
        source = "step4_saved"
    else:
        if target_name not in raw_name_to_idx:
            return None
        row = r2_raw_df[r2_raw_df["target"] == target_name]
        if row.empty:
            return None
        row = row.iloc[0]
        w_raw = np.asarray(directions_raw_npz["w"][raw_name_to_idx[target_name]], dtype=np.float32)
        source = "step4_saved"

    w_std = coef_raw_to_std(w_raw)
    return {
        "target": target_name,
        "alpha": float(row["alpha"]) if "alpha" in row.index and pd.notna(row["alpha"]) else np.nan,
        "target_mean_train": float(row["target_mean_train"]),
        "target_std_train": float(row["target_std_train"]),
        "w_raw": w_raw,
        "w_std": w_std,
        "direction_std_unit": unit_vector(w_std),
        "w_std_norm": float(np.linalg.norm(w_std)),
        "r2_train": float(row["r2_train"]) if "r2_train" in row.index and pd.notna(row["r2_train"]) else np.nan,
        "r2_val": float(row["r2_val"]) if "r2_val" in row.index and pd.notna(row["r2_val"]) else np.nan,
        "r2_test": float(row["r2_test"]) if "r2_test" in row.index and pd.notna(row["r2_test"]) else np.nan,
        "mae_train": float(row["mae_train"]) if "mae_train" in row.index and pd.notna(row["mae_train"]) else np.nan,
        "mae_val": float(row["mae_val"]) if "mae_val" in row.index and pd.notna(row["mae_val"]) else np.nan,
        "mae_test": float(row["mae_test"]) if "mae_test" in row.index and pd.notna(row["mae_test"]) else np.nan,
        "source": source,
    }

def build_confound_bundle(confound_name):
    conf_values = pd.to_numeric(panel[confound_name], errors="coerce").to_numpy(dtype=float)
    fitted = fit_probe_single(confound_name, conf_values)
    if confound_name in saved_confound_raw_map:
        w_raw = np.asarray(saved_confound_raw_map[confound_name], dtype=np.float32)
        w_std = coef_raw_to_std(w_raw)
        fitted.update({
            "w_raw": w_raw,
            "w_std": w_std,
            "direction_std_unit": unit_vector(w_std),
            "w_std_norm": float(np.linalg.norm(w_std)),
            "source": "step3_saved_direction",
        })
    return fitted

def build_random_bundle(name_for_seed, reference_probe):
    if step4_random_unit_dirs is not None and len(step4_random_unit_dirs) > 0:
        idx = stable_random_index(name_for_seed, len(step4_random_unit_dirs))
        w_raw = np.asarray(step4_random_unit_dirs[idx], dtype=np.float32)
        source = "step4_random_controls"
        random_index = idx
    else:
        rng = np.random.default_rng(SEED + stable_random_index(name_for_seed, 100000))
        w_raw = unit_vector(rng.normal(size=z_raw.shape[1]).astype(np.float32))
        source = "inline_random"
        random_index = np.nan
    w_std = coef_raw_to_std(w_raw)
    return {
        "target": PROPERTY,
        "alpha": np.nan,
        "target_mean_train": float(reference_probe["target_mean_train"]),
        "target_std_train": float(reference_probe["target_std_train"]),
        "w_raw": w_raw,
        "w_std": w_std,
        "direction_std_unit": unit_vector(w_std),
        "w_std_norm": float(np.linalg.norm(w_std)),
        "r2_train": np.nan,
        "r2_val": np.nan,
        "r2_test": np.nan,
        "mae_train": np.nan,
        "mae_val": np.nan,
        "mae_test": np.nan,
        "source": source,
        "random_index": random_index,
    }

property_values = pd.to_numeric(panel[PROPERTY], errors="coerce").to_numpy(dtype=float)
confound_fit = fit_property_residualizer(PROPERTY)
resid_property_values = confound_fit["resid_all"]

raw_probe = load_saved_probe(PROPERTY, residual=False)
if raw_probe is None:
    if not ALLOW_INLINE_FALLBACK:
        raise FileNotFoundError(f"Could not load saved Step 4 raw direction for {PROPERTY}.")
    raw_probe = fit_probe_single(PROPERTY, property_values)

resid_probe = load_saved_probe(PROPERTY_RESID_NAME, residual=True)
if resid_probe is None:
    if not ALLOW_INLINE_FALLBACK:
        raise FileNotFoundError(f"Could not load saved Step 4 residual direction for {PROPERTY_RESID_NAME}.")
    resid_probe = fit_probe_single(PROPERTY_RESID_NAME, resid_property_values)

confound_bundles = {name: build_confound_bundle(name) for name in C_COLS}

if STEP4_AVAILABLE and not cos_raw_df.empty and PROPERTY in set(cos_raw_df["target"].astype(str)):
    confound_row = cos_raw_df[cos_raw_df["target"].astype(str) == PROPERTY].iloc[0]
    linked_confound = max(C_COLS, key=lambda c: abs(float(confound_row[c])))
else:
    linked_confound = max(
        C_COLS,
        key=lambda c: abs(cosine(raw_probe["direction_std_unit"], confound_bundles[c]["direction_std_unit"]))
    )

linked_confound_bundle = confound_bundles[linked_confound]
random_probe = build_random_bundle(PROPERTY, raw_probe)

alignment_rows = []
for confound_name, bundle in confound_bundles.items():
    alignment_rows.append({
        "confound": confound_name,
        "raw_vs_confound_cosine": cosine(raw_probe["direction_std_unit"], bundle["direction_std_unit"]),
        "resid_vs_confound_cosine": cosine(resid_probe["direction_std_unit"], bundle["direction_std_unit"]),
        "confound_direction_source": bundle["source"],
        "confound_r2_val": bundle["r2_val"],
    })
alignment_df = pd.DataFrame(alignment_rows).sort_values(
    "raw_vs_confound_cosine",
    key=lambda s: s.abs(),
    ascending=False,
).reset_index(drop=True)

probe_summary_df = pd.DataFrame([
    {
        "model": "C_to_PROPERTY",
        "target": PROPERTY,
        "alpha": np.nan,
        "target_mean_train": float(np.nanmean(property_values[train_mask])),
        "target_std_train": float(np.nanstd(property_values[train_mask])),
        "r2_train": confound_fit["r2_train"],
        "r2_val": confound_fit["r2_val"],
        "r2_test": confound_fit["r2_test"],
        "mae_train": confound_fit["mae_train"],
        "mae_val": confound_fit["mae_val"],
        "mae_test": confound_fit["mae_test"],
        "source": "inline_confound_residualizer",
    },
    {
        "model": "Z_to_PROPERTY",
        "target": PROPERTY,
        "alpha": raw_probe["alpha"],
        "target_mean_train": raw_probe["target_mean_train"],
        "target_std_train": raw_probe["target_std_train"],
        "r2_train": raw_probe["r2_train"],
        "r2_val": raw_probe["r2_val"],
        "r2_test": raw_probe["r2_test"],
        "mae_train": raw_probe["mae_train"],
        "mae_val": raw_probe["mae_val"],
        "mae_test": raw_probe["mae_test"],
        "source": raw_probe["source"],
    },
    {
        "model": "Z_to_resid_PROPERTY",
        "target": PROPERTY_RESID_NAME,
        "alpha": resid_probe["alpha"],
        "target_mean_train": resid_probe["target_mean_train"],
        "target_std_train": resid_probe["target_std_train"],
        "r2_train": resid_probe["r2_train"],
        "r2_val": resid_probe["r2_val"],
        "r2_test": resid_probe["r2_test"],
        "mae_train": resid_probe["mae_train"],
        "mae_val": resid_probe["mae_val"],
        "mae_test": resid_probe["mae_test"],
        "source": resid_probe["source"],
    },
    {
        "model": "Z_to_linked_confound",
        "target": linked_confound,
        "alpha": linked_confound_bundle["alpha"],
        "target_mean_train": linked_confound_bundle["target_mean_train"],
        "target_std_train": linked_confound_bundle["target_std_train"],
        "r2_train": linked_confound_bundle["r2_train"],
        "r2_val": linked_confound_bundle["r2_val"],
        "r2_test": linked_confound_bundle["r2_test"],
        "mae_train": linked_confound_bundle["mae_train"],
        "mae_val": linked_confound_bundle["mae_val"],
        "mae_test": linked_confound_bundle["mae_test"],
        "source": linked_confound_bundle["source"],
    },
    {
        "model": "random_control",
        "target": PROPERTY,
        "alpha": np.nan,
        "target_mean_train": random_probe["target_mean_train"],
        "target_std_train": random_probe["target_std_train"],
        "r2_train": np.nan,
        "r2_val": np.nan,
        "r2_test": np.nan,
        "mae_train": np.nan,
        "mae_val": np.nan,
        "mae_test": np.nan,
        "source": random_probe["source"],
    },
])

residual_alignment_rows = []
if PROPERTY_RESID_NAME in panel.columns:
    saved_resid = pd.to_numeric(panel[PROPERTY_RESID_NAME], errors="coerce")
    inline_resid = pd.Series(resid_property_values, dtype=float)
    residual_alignment_rows.append({
        "metric": "corr(saved_step3_resid, inline_resid)",
        "value": float(saved_resid.corr(inline_resid)) if saved_resid.notna().sum() > 0 and inline_resid.notna().sum() > 0 else np.nan,
    })
residual_alignment_rows.append({"metric": "linked_confound_raw_probe", "value": linked_confound})
resid_alignment_df = pd.DataFrame(residual_alignment_rows)

probe_summary_csv = TABLES_DIR / f"{PROPERTY.lower()}_probe_summary.csv"
alignment_csv = TABLES_DIR / f"{PROPERTY.lower()}_direction_alignment.csv"
resid_alignment_csv = TABLES_DIR / f"{PROPERTY.lower()}_residual_alignment.csv"

probe_summary_df.to_csv(probe_summary_csv, index=False)
alignment_df.to_csv(alignment_csv, index=False)
resid_alignment_df.to_csv(resid_alignment_csv, index=False)

print("STEP4_AVAILABLE:", STEP4_AVAILABLE)
print("linked_confound:", linked_confound)
print("Saved:", probe_summary_csv)
print("Saved:", alignment_csv)
print("Saved:", resid_alignment_csv)
display(probe_summary_df)
display(alignment_df)
display(resid_alignment_df)

## 5. Recompute decoded chemistry, confounds, and the residualized target

In [ ]:
def shannon_entropy(tokens):
    if len(tokens) == 0:
        return 0.0
    counts = pd.Series(tokens).value_counts().to_numpy(dtype=float)
    p = counts / counts.sum()
    return float(-(p * np.log(p)).sum())

def safe_float(fn):
    try:
        return float(fn())
    except Exception:
        return np.nan

MORGAN_GENERATOR = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def make_morgan_fp(mol):
    if mol is None:
        return None
    return MORGAN_GENERATOR.GetFingerprint(mol)

def tanimoto_similarity(fp_a, fp_b):
    if fp_a is None or fp_b is None:
        return np.nan
    return float(DataStructs.TanimotoSimilarity(fp_a, fp_b))

def tokenize_selfies_string(selfies_str):
    if not selfies_str:
        return []
    try:
        return list(sf.split_selfies(selfies_str))
    except Exception:
        return []

def compute_confound_features(selfies_str):
    tokens = tokenize_selfies_string(selfies_str)
    return {
        "selfies_len_tokens": float(len(tokens)),
        "branch_token_count": float(sum("Branch" in tok for tok in tokens)),
        "ring_token_count": float(sum("Ring" in tok for tok in tokens)),
        "token_entropy": shannon_entropy(tokens),
    }

def compute_property_panel_for_mol(mol):
    empty = {name: np.nan for name in Y_COLS}
    counts = {
        "carbon_count": np.nan,
        "heteroatom_count": np.nan,
    }
    if mol is None:
        return {**empty, **counts}

    carbon_count = int(sum(atom.GetAtomicNum() == 6 for atom in mol.GetAtoms()))
    heteroatom_count = int(sum(atom.GetAtomicNum() not in (1, 6) for atom in mol.GetAtoms()))

    values = {
        "MolWt": safe_float(lambda: Descriptors.MolWt(mol)),
        "ExactMolWt": safe_float(lambda: Descriptors.ExactMolWt(mol)),
        "HeavyAtomCount": safe_float(lambda: float(mol.GetNumHeavyAtoms())),
        "cLogP": safe_float(lambda: Crippen.MolLogP(mol)),
        "TPSA": safe_float(lambda: rdMolDescriptors.CalcTPSA(mol)),
        "HBD": safe_float(lambda: rdMolDescriptors.CalcNumHBD(mol)),
        "HBA": safe_float(lambda: rdMolDescriptors.CalcNumHBA(mol)),
        "NumRotatableBonds": safe_float(lambda: rdMolDescriptors.CalcNumRotatableBonds(mol)),
        "RingCount": safe_float(lambda: rdMolDescriptors.CalcNumRings(mol)),
        "AromaticRingCount": safe_float(lambda: rdMolDescriptors.CalcNumAromaticRings(mol)),
        "FractionCSP3": safe_float(lambda: rdMolDescriptors.CalcFractionCSP3(mol)),
        "NumSpiroAtoms": safe_float(lambda: rdMolDescriptors.CalcNumSpiroAtoms(mol)),
        "NumBridgeheadAtoms": safe_float(lambda: rdMolDescriptors.CalcNumBridgeheadAtoms(mol)),
        "BertzCT": safe_float(lambda: Descriptors.BertzCT(mol)),
        "QED": safe_float(lambda: QED.qed(mol)),
    }
    return {
        **values,
        "carbon_count": float(carbon_count),
        "heteroatom_count": float(heteroatom_count),
    }

def compute_raw_property_features(smiles_str):
    mol = smiles_to_mol(smiles_str)
    fp = make_morgan_fp(mol)
    features = compute_property_panel_for_mol(mol)
    return features, mol, fp

def apply_property_residualizer(feature_df):
    out = feature_df.copy()
    conf_mat = out[C_COLS].to_numpy(dtype=float)
    pred = np.full(len(out), np.nan, dtype=float)

    valid = np.isfinite(conf_mat).all(axis=1)
    if valid.any():
        pred_s = confound_fit["model"].predict(confound_fit["c_scaler"].transform(conf_mat[valid]))
        pred[valid] = confound_fit["y_scaler"].inverse_transform(np.asarray(pred_s).reshape(-1, 1)).ravel()

    out[f"pred_from_confounds_{PROPERTY}"] = pred
    out[PROPERTY_RESID_NAME] = pd.to_numeric(out[PROPERTY], errors="coerce").to_numpy(dtype=float) - pred
    return out

def build_candidate_features(decoded_pack_df, seed_fp):
    feature_rows = []
    fps = []
    for row in decoded_pack_df.itertuples(index=False):
        raw_features, mol, fp = compute_raw_property_features(row.smiles)
        confound_features = compute_confound_features(row.selfies)
        combined = {**raw_features, **confound_features, "seed_similarity": tanimoto_similarity(fp, seed_fp)}
        feature_rows.append(combined)
        fps.append(fp)
    feature_df = pd.DataFrame(feature_rows)
    feature_df = apply_property_residualizer(feature_df)
    feature_df["candidate_fp"] = fps
    return feature_df

## 6. Traversal geometry and corrected free-length / fixed-length decoding

In [ ]:
def sample_pair_distances_std(z_std_latents, n_pairs, seed):
    rng = np.random.default_rng(int(seed))
    n_val = int(len(z_std_latents))
    idx1 = rng.integers(0, n_val, size=n_pairs)
    idx2 = rng.integers(0, n_val, size=n_pairs)
    same = idx1 == idx2
    while np.any(same):
        idx2[same] = rng.integers(0, n_val, size=int(np.sum(same)))
        same = idx1 == idx2
    distances = np.linalg.norm(z_std_latents[idx2] - z_std_latents[idx1], axis=1)
    return distances.astype(np.float32)

benchmark_total_span_std_ref = float(np.median(sample_pair_distances_std(z_std_all[val_mask], BENCHMARK_PAIR_COUNT, SEED)))
reference_step_span = float(STEP_GRID.max() - STEP_GRID.min())
DEFAULT_TOTAL_SPAN_STD = float(DEFAULT_TOTAL_SPAN_FACTOR_OF_BENCHMARK * benchmark_total_span_std_ref)
MAX_TOTAL_SPAN_STD = float(MAX_TOTAL_SPAN_FACTOR_OF_BENCHMARK * benchmark_total_span_std_ref)
SWEEP_MAX_TOTAL_SPAN_STD = float(SPAN_SWEEP_MAX_FACTOR_OF_BENCHMARK * benchmark_total_span_std_ref)

def format_span_label(multiplier):
    multiplier = float(multiplier)
    if abs(multiplier - round(multiplier)) < 1e-9:
        return f"x{int(round(multiplier))}"
    return f"x{multiplier:g}"

SPAN_SWEEP_LABELS = [format_span_label(x) for x in SPAN_SWEEP_MULTIPLIERS]
STRIP_SPAN_LABEL = format_span_label(STRIP_SPAN_MULTIPLIER)

def build_direction_specs(variant, bundle, target_name):
    delta_prop_nominal = TARGET_STEP_FRACTION_OF_STD * float(bundle["target_std_train"]) if np.isfinite(bundle["target_std_train"]) else np.nan
    w_std_norm = float(bundle["w_std_norm"])
    delta_step_std_nominal = delta_prop_nominal / w_std_norm if np.isfinite(delta_prop_nominal) and w_std_norm > 1e-12 else np.nan
    total_span_nominal = delta_step_std_nominal * reference_step_span if np.isfinite(delta_step_std_nominal) else np.nan

    if np.isfinite(total_span_nominal) and total_span_nominal > 0:
        target_total_span_std = float(min(total_span_nominal, MAX_TOTAL_SPAN_STD))
    else:
        target_total_span_std = float(DEFAULT_TOTAL_SPAN_STD)

    specs = []
    for span_multiplier in SPAN_SWEEP_MULTIPLIERS:
        span_multiplier = float(span_multiplier)
        span_label = format_span_label(span_multiplier)
        swept_total_span_std = float(min(target_total_span_std * span_multiplier, SWEEP_MAX_TOTAL_SPAN_STD))
        delta_step_std_effective = float(swept_total_span_std / reference_step_span) if reference_step_span > 0 else np.nan
        specs.append({
            "direction_variant": variant,
            "span_multiplier": span_multiplier,
            "span_label": span_label,
            "direction_label": f"{PROPERTY}__{variant}__{span_label}",
            "direction_space": "standardized_latent",
            "target_name": target_name,
            "confound_name": linked_confound,
            "direction_source": bundle["source"],
            "target_mean_train": float(bundle["target_mean_train"]),
            "target_std_train": float(bundle["target_std_train"]),
            "w_raw": np.asarray(bundle["w_raw"], dtype=np.float32),
            "w_std_norm": float(bundle["w_std_norm"]),
            "direction_unit_std": np.asarray(bundle["direction_std_unit"], dtype=np.float32),
            "delta_prop_nominal": float(delta_prop_nominal) if np.isfinite(delta_prop_nominal) else np.nan,
            "delta_step_std_nominal": float(delta_step_std_nominal) if np.isfinite(delta_step_std_nominal) else np.nan,
            "delta_step_std_effective": float(delta_step_std_effective) if np.isfinite(delta_step_std_effective) else np.nan,
            "base_target_total_span_std": float(target_total_span_std),
            "target_total_span_std": float(swept_total_span_std),
            "step_values_std": (STEP_GRID.astype(float) * float(delta_step_std_effective)).tolist() if np.isfinite(delta_step_std_effective) else [0.0] * len(STEP_GRID),
            "benchmark_total_span_std_ref": float(benchmark_total_span_std_ref),
            "sweep_max_total_span_std": float(SWEEP_MAX_TOTAL_SPAN_STD),
            "random_index": bundle.get("random_index", np.nan),
        })
    return specs

direction_specs = [
    *build_direction_specs("raw", raw_probe, PROPERTY),
    *build_direction_specs("residual", resid_probe, PROPERTY_RESID_NAME),
    *build_direction_specs("confound", linked_confound_bundle, linked_confound),
    *build_direction_specs("random", random_probe, PROPERTY),
]
direction_specs_df = pd.DataFrame([
    {k: v for k, v in spec.items() if k not in {"w_raw", "direction_unit_std", "step_values_std"}}
    | {"step_values_std": json.dumps(spec["step_values_std"])}
    for spec in direction_specs
])
direction_specs_csv = TABLES_DIR / f"{PROPERTY.lower()}_direction_specs.csv"
direction_specs_df.to_csv(direction_specs_csv, index=False)

print("benchmark_total_span_std_ref:", round(benchmark_total_span_std_ref, 6))
print("default_total_span_std:", round(DEFAULT_TOTAL_SPAN_STD, 6))
print("max_total_span_std:", round(MAX_TOTAL_SPAN_STD, 6))
print("sweep_max_total_span_std:", round(SWEEP_MAX_TOTAL_SPAN_STD, 6))
print("Saved:", direction_specs_csv)
display(direction_specs_df)

def unpack_decoder_output(decoder_out):
    if isinstance(decoder_out, (tuple, list)):
        logits = decoder_out[0]
        pred_len = decoder_out[1] if len(decoder_out) > 1 else None
    else:
        logits = decoder_out
        pred_len = None
    return logits, pred_len

def trim_pred_ids(token_ids, pred_len_value=None):
    ids = [int(x) for x in token_ids]
    if pred_len_value is not None:
        pred_len_value = max(1, int(round(float(pred_len_value))))
        ids = ids[:pred_len_value]
    return ids

def decode_latents(z_batch, forced_len=None, sos_id=SOS_ID):
    z_batch = torch.as_tensor(z_batch, dtype=torch.float32, device=device)
    if z_batch.dim() == 1:
        z_batch = z_batch.unsqueeze(0)

    with torch.no_grad():
        if forced_len is None:
            decoder_out = model.decode(z_batch, mode="eval", sos_id=sos_id)
            logits, pred_len = unpack_decoder_output(decoder_out)
        else:
            B, _ = z_batch.shape
            forced_len_i = torch.full((B,), max(1, int(forced_len)), dtype=torch.long, device=z_batch.device)
            pred_len = forced_len_i.float()

            z_proj = model.fc_z2h(z_batch)
            slots = z_proj.view(B, model.num_slots, model.hidden_size)
            slots = slots - slots.mean(dim=1, keepdim=True)
            slots = F.layer_norm(slots, slots.shape[-1:])

            max_len_batch = int(forced_len_i.max().item())
            t = torch.arange(max_len_batch, device=z_batch.device)[None, :]

            pos_q = model.pos_encoder(torch.empty(B, max_len_batch, model.hidden_size, device=z_batch.device))
            pos_q = pos_q.expand(B, -1, -1).detach().clone()
            pos_q[:, 0, :] = model.embedding.weight[sos_id]

            h = model.cross_block(pos_q, slots, slots)
            h = F.layer_norm(h, h.shape[-1:])

            dec_mask = t < forced_len_i[:, None]
            for block in model.decoder_blocks:
                h = block(h, h, h, pad_mask=dec_mask)

            logits = model.fc_output(h)

    pred_ids = logits.argmax(dim=-1)
    rows = []
    for i in range(pred_ids.size(0)):
        raw_length_value = None if pred_len is None else float(pred_len[i].item())
        used_len = None if raw_length_value is None else max(1, int(round(raw_length_value)))
        ids = trim_pred_ids(pred_ids[i].detach().cpu().tolist(), used_len)
        selfies_str = decode_tokens_to_selfies(ids, id2tok, PAD_ID, SOS_ID, EOS_ID)
        smiles_str = selfies_to_smiles(selfies_str)
        canonical = canon_smiles(smiles_str)
        rows.append({
            "pred_ids": ids,
            "selfies": selfies_str,
            "smiles": smiles_str,
            "canonical_smiles": canonical,
            "valid": int(canonical is not None),
            "pred_len_value": raw_length_value,
            "pred_len_used": used_len if used_len is not None else len(ids),
            "forced_len": None if forced_len is None else int(forced_len),
            "decode_mode_effective": "free_length" if forced_len is None else "fixed_length_seed_pred",
        })
    return pd.DataFrame(rows)

## 7. Continuity-aware path selection and traversal metrics

In [ ]:
def orthogonal_noise_std(base_direction_std, sigma_std, n_samples, rng_seed):
    direction = np.asarray(base_direction_std, dtype=np.float32)
    direction_norm = float(np.linalg.norm(direction))
    if n_samples <= 0 or sigma_std <= 0 or direction_norm < 1e-8:
        return np.zeros((max(n_samples, 0), direction.shape[0]), dtype=np.float32)
    unit = direction / direction_norm
    rng_local = np.random.default_rng(int(rng_seed))
    noise = rng_local.normal(size=(n_samples, direction.shape[0])).astype(np.float32)
    proj = (noise @ unit)[:, None] * unit[None, :]
    noise = noise - proj
    norms = np.linalg.norm(noise, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-8, None)
    return sigma_std * noise / norms

def within_step_medoid(decoded_step_df):
    valid = decoded_step_df[decoded_step_df["valid"] == 1].copy()
    if valid.empty:
        row = decoded_step_df.iloc[0].copy()
        row["medoid_valid_count"] = 0
        row["medoid_unique_valid_count"] = 0
        row["medoid_avg_distance"] = np.nan
        row["medoid_candidate_index"] = int(row["candidate_index"])
        return row

    fps = list(valid["candidate_fp"])
    if len(valid) == 1:
        row = valid.iloc[0].copy()
        row["medoid_valid_count"] = 1
        row["medoid_unique_valid_count"] = 1
        row["medoid_avg_distance"] = 0.0
        row["medoid_candidate_index"] = int(row["candidate_index"])
        return row

    avg_dist = []
    for i, fp_i in enumerate(fps):
        distances = []
        for j, fp_j in enumerate(fps):
            if i == j:
                continue
            distances.append(1.0 - DataStructs.TanimotoSimilarity(fp_i, fp_j))
        avg_dist.append(float(np.mean(distances)))

    medoid_local = int(np.argmin(avg_dist))
    row = valid.iloc[medoid_local].copy()
    row["medoid_valid_count"] = int(len(valid))
    row["medoid_unique_valid_count"] = int(valid["canonical_smiles"].nunique())
    row["medoid_avg_distance"] = float(avg_dist[medoid_local])
    row["medoid_candidate_index"] = int(row["candidate_index"])
    return row

def candidate_node_cost(row):
    if int(row["valid"]) != 1:
        return 100.0
    medoid_avg_distance = row.get("medoid_avg_distance", np.nan)
    if not np.isfinite(medoid_avg_distance):
        medoid_avg_distance = 0.0
    seed_similarity = row.get("seed_similarity", np.nan)
    anchor = 0.25 * (1.0 - seed_similarity) if np.isfinite(seed_similarity) else 0.25
    return float(medoid_avg_distance + anchor)

def transition_cost(prev_row, curr_row):
    if int(prev_row["valid"]) == 1 and int(curr_row["valid"]) == 1:
        fp_prev = prev_row["candidate_fp"]
        fp_curr = curr_row["candidate_fp"]
        if fp_prev is not None and fp_curr is not None:
            return float(1.0 - DataStructs.TanimotoSimilarity(fp_prev, fp_curr))
    return 2.0

def dynamic_programming_path(step_candidate_groups):
    states = []
    backpointers = []
    for step_idx, step_df in enumerate(step_candidate_groups):
        costs = {}
        back = {}
        if step_idx == 0:
            for row in step_df.itertuples(index=False):
                costs[int(row.candidate_index)] = candidate_node_cost(row._asdict())
                back[int(row.candidate_index)] = None
        else:
            prev_df = step_candidate_groups[step_idx - 1]
            prev_costs = states[-1]
            for row in step_df.itertuples(index=False):
                curr_idx = int(row.candidate_index)
                curr_dict = row._asdict()
                curr_node_cost = candidate_node_cost(curr_dict)
                best_cost = np.inf
                best_prev = None
                for prev_row in prev_df.itertuples(index=False):
                    prev_idx = int(prev_row.candidate_index)
                    total = prev_costs[prev_idx] + transition_cost(prev_row._asdict(), curr_dict) + curr_node_cost
                    if total < best_cost:
                        best_cost = total
                        best_prev = prev_idx
                costs[curr_idx] = float(best_cost)
                back[curr_idx] = best_prev
        states.append(costs)
        backpointers.append(back)

    final_costs = states[-1]
    end_idx = min(final_costs, key=final_costs.get)
    path = [end_idx]
    for step_idx in range(len(step_candidate_groups) - 1, 0, -1):
        path.append(backpointers[step_idx][path[-1]])
    path.reverse()
    return path, float(final_costs[end_idx])

def greedy_medoid_path_cost(step_candidate_groups):
    medoid_indices = []
    total_cost = 0.0
    prev_row = None
    for step_df in step_candidate_groups:
        medoid_row = within_step_medoid(step_df)
        medoid_dict = medoid_row.to_dict()
        medoid_indices.append(int(medoid_dict["candidate_index"]))
        total_cost += candidate_node_cost(medoid_dict)
        if prev_row is not None:
            total_cost += transition_cost(prev_row, medoid_dict)
        prev_row = medoid_dict
    return medoid_indices, float(total_cost)

def monotonic_violation_count(values):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size < 2:
        return np.nan
    diffs = np.diff(arr)
    positive = int(np.sum(diffs > 0))
    negative = int(np.sum(diffs < 0))
    if positive >= negative:
        return int(np.sum(diffs < 0))
    return int(np.sum(diffs > 0))

def spearman_corr_steps(step_values, y_values):
    x = pd.Series(step_values, dtype=float)
    y = pd.Series(y_values, dtype=float)
    mask = x.notna() & y.notna()
    if int(mask.sum()) < 2:
        return np.nan
    return float(x[mask].corr(y[mask], method="spearman"))

def linear_slope(step_values, y_values):
    x = np.asarray(step_values, dtype=float)
    y = np.asarray(y_values, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if int(mask.sum()) < 2:
        return np.nan
    slope, _ = np.polyfit(x[mask], y[mask], 1)
    return float(slope)

def adjacent_tanimoto_series(path_df):
    values = []
    prev_fp = None
    prev_valid = None
    for row in path_df.itertuples(index=False):
        fp = row.candidate_fp
        if prev_fp is not None and fp is not None and prev_valid == 1 and int(row.valid) == 1:
            values.append(float(DataStructs.TanimotoSimilarity(prev_fp, fp)))
        else:
            values.append(np.nan)
        prev_fp = fp
        prev_valid = int(row.valid)
    return values

def structural_consistency_fraction(path_df, col):
    values = pd.to_numeric(path_df[col], errors="coerce").to_numpy(dtype=float)
    if len(values) < 2 or not np.isfinite(values[0]) or not np.isfinite(values[-1]):
        return np.nan
    net_delta = values[-1] - values[0]
    if abs(net_delta) < 1e-12:
        return np.nan
    diffs = np.diff(values)
    nonzero = diffs[np.isfinite(diffs) & (np.abs(diffs) > 1e-12)]
    if nonzero.size == 0:
        return np.nan
    return float(np.mean(np.sign(nonzero) == np.sign(net_delta)))

def add_path_level_metrics(path_df, seed_row):
    out = path_df.copy()
    out["adjacent_step_tanimoto"] = adjacent_tanimoto_series(out)
    out["delta_heavy_atom_count"] = pd.to_numeric(out["HeavyAtomCount"], errors="coerce").diff()
    out["delta_carbon_count"] = pd.to_numeric(out["carbon_count"], errors="coerce").diff()
    out["delta_heteroatom_count"] = pd.to_numeric(out["heteroatom_count"], errors="coerce").diff()
    out["delta_ring_count"] = pd.to_numeric(out["RingCount"], errors="coerce").diff()
    out["delta_aromatic_ring_count"] = pd.to_numeric(out["AromaticRingCount"], errors="coerce").diff()
    out["seed_band"] = seed_row["seed_band"]
    out["seed_raw_property"] = seed_row["seed_raw_property"]
    out["seed_canonical_smiles"] = seed_row["canonical_smiles"]
    return out

def summarize_seed_run(candidate_df, path_df, seed_row, spec_row):
    metric_target_name = spec_row["target_name"]
    confound_name = spec_row["confound_name"]

    step_stats = candidate_df.groupby("step_index").agg(
        validity_fraction=("valid", "mean"),
        valid_count=("valid", "sum"),
        candidate_count=("valid", "size"),
        unique_valid=("canonical_smiles", lambda s: pd.Series([x for x in s if pd.notna(x)]).nunique()),
        median_pred_len_used=("pred_len_used", "median"),
        median_target=(metric_target_name, "median"),
        median_residual=(PROPERTY_RESID_NAME, "median"),
        median_confound=(confound_name, "median"),
        median_seed_similarity=("seed_similarity", "median"),
    ).reset_index()
    step_stats["uniqueness_fraction"] = np.where(
        step_stats["valid_count"] > 0,
        step_stats["unique_valid"] / step_stats["valid_count"],
        np.nan,
    )

    path_steps = pd.to_numeric(path_df["step_value_std"], errors="coerce").to_numpy(dtype=float)
    target_values = pd.to_numeric(path_df[metric_target_name], errors="coerce").to_numpy(dtype=float)
    confound_values = pd.to_numeric(path_df[confound_name], errors="coerce").to_numpy(dtype=float)
    pred_len_values = pd.to_numeric(path_df["pred_len_used"], errors="coerce").to_numpy(dtype=float)

    summary = {
        "property": PROPERTY,
        "base_property": PROPERTY,
        "direction_variant": spec_row["direction_variant"],
        "direction_label": spec_row["direction_label"],
        "direction_source": spec_row["direction_source"],
        "direction_space": spec_row["direction_space"],
        "decode_mode": path_df["decode_mode"].iloc[0],
        "seed_index": int(seed_row["seed_index"]),
        "seed_band": seed_row["seed_band"],
        "seed_canonical_smiles": seed_row["canonical_smiles"],
        "seed_raw_property": float(seed_row["seed_raw_property"]),
        "target_metric_name": metric_target_name,
        "confound_name": confound_name,
        "delta_prop_nominal": float(spec_row["delta_prop_nominal"]) if np.isfinite(spec_row["delta_prop_nominal"]) else np.nan,
        "delta_step_std_nominal": float(spec_row["delta_step_std_nominal"]) if np.isfinite(spec_row["delta_step_std_nominal"]) else np.nan,
        "delta_step_std_effective": float(spec_row["delta_step_std_effective"]),
        "target_total_span_std": float(spec_row["target_total_span_std"]),
        "predicted_delta_prop_path_end": float(pd.to_numeric(path_df["predicted_delta_prop_effective"], errors="coerce").iloc[-1]) if "predicted_delta_prop_effective" in path_df.columns else np.nan,
        "pack_size": int(candidate_df["candidate_index"].nunique() / max(candidate_df["step_index"].nunique(), 1)),
        "step_count": int(path_df["step_index"].nunique()),
        "dp_path_total_cost": float(path_df["dp_path_total_cost"].iloc[0]),
        "greedy_medoid_total_cost": float(path_df["greedy_medoid_total_cost"].iloc[0]),
        "dp_beats_or_matches_medoid": float(path_df["dp_path_total_cost"].iloc[0] <= path_df["greedy_medoid_total_cost"].iloc[0] + 1e-9),
        "target_spearman": spearman_corr_steps(path_steps, target_values),
        "target_slope": linear_slope(path_steps, target_values),
        "target_monotonic_violation_count": monotonic_violation_count(target_values),
        "residual_target_slope": linear_slope(path_steps, path_df[PROPERTY_RESID_NAME]),
        "validity_fraction": float(candidate_df["valid"].mean()),
        "uniqueness_fraction_valid": float(candidate_df.loc[candidate_df["valid"] == 1, "canonical_smiles"].nunique() / max(int((candidate_df["valid"] == 1).sum()), 1)),
        "path_unique_canonical_count": int(path_df["canonical_smiles"].dropna().nunique()),
        "path_constant_across_steps": float(path_df["canonical_smiles"].dropna().nunique() <= 1),
        "path_similarity_to_seed_mean": float(pd.to_numeric(path_df["seed_similarity"], errors="coerce").mean()),
        "path_similarity_to_seed_end": float(pd.to_numeric(path_df["seed_similarity"], errors="coerce").iloc[-1]),
        "adjacent_step_tanimoto_mean": float(pd.to_numeric(path_df["adjacent_step_tanimoto"], errors="coerce").mean()),
        "adjacent_step_tanimoto_median": float(pd.to_numeric(path_df["adjacent_step_tanimoto"], errors="coerce").median()),
        "confound_spearman": spearman_corr_steps(path_steps, confound_values),
        "confound_slope": linear_slope(path_steps, confound_values),
        "confound_delta_end_start": float(confound_values[-1] - confound_values[0]) if np.isfinite(confound_values[0]) and np.isfinite(confound_values[-1]) else np.nan,
        "pred_len_spearman": spearman_corr_steps(path_steps, pred_len_values),
        "pred_len_slope": linear_slope(path_steps, pred_len_values),
        "pred_len_delta_end_start": float(pred_len_values[-1] - pred_len_values[0]) if np.isfinite(pred_len_values[0]) and np.isfinite(pred_len_values[-1]) else np.nan,
        "selfies_len_tokens_slope": linear_slope(path_steps, path_df["selfies_len_tokens"]),
        "ring_token_count_slope": linear_slope(path_steps, path_df["ring_token_count"]),
        "branch_token_count_slope": linear_slope(path_steps, path_df["branch_token_count"]),
        "token_entropy_slope": linear_slope(path_steps, path_df["token_entropy"]),
        "heavy_atom_delta_end_start": float(pd.to_numeric(path_df["HeavyAtomCount"], errors="coerce").iloc[-1] - pd.to_numeric(path_df["HeavyAtomCount"], errors="coerce").iloc[0]),
        "carbon_delta_end_start": float(pd.to_numeric(path_df["carbon_count"], errors="coerce").iloc[-1] - pd.to_numeric(path_df["carbon_count"], errors="coerce").iloc[0]),
        "heteroatom_delta_end_start": float(pd.to_numeric(path_df["heteroatom_count"], errors="coerce").iloc[-1] - pd.to_numeric(path_df["heteroatom_count"], errors="coerce").iloc[0]),
        "ring_delta_end_start": float(pd.to_numeric(path_df["RingCount"], errors="coerce").iloc[-1] - pd.to_numeric(path_df["RingCount"], errors="coerce").iloc[0]),
        "aromatic_ring_delta_end_start": float(pd.to_numeric(path_df["AromaticRingCount"], errors="coerce").iloc[-1] - pd.to_numeric(path_df["AromaticRingCount"], errors="coerce").iloc[0]),
        "heavy_atom_consistency_fraction": structural_consistency_fraction(path_df, "HeavyAtomCount"),
        "carbon_consistency_fraction": structural_consistency_fraction(path_df, "carbon_count"),
        "heteroatom_consistency_fraction": structural_consistency_fraction(path_df, "heteroatom_count"),
        "ring_consistency_fraction": structural_consistency_fraction(path_df, "RingCount"),
        "aromatic_ring_consistency_fraction": structural_consistency_fraction(path_df, "AromaticRingCount"),
    }
    return summary, step_stats

def select_property_seeds(val_panel_df, property_name, seed_count):
    sub = val_panel_df.copy()
    sub[property_name] = pd.to_numeric(sub[property_name], errors="coerce")
    sub = sub[sub["canonical_smiles"].notna() & sub[property_name].notna()].copy()
    sub = sub.sort_values([property_name, "canonical_smiles", "dataset_index"]).drop_duplicates("canonical_smiles", keep="first").reset_index(drop=True)
    if sub.empty:
        return sub

    values = sub[property_name].to_numpy(dtype=float)
    q1, q2 = np.quantile(values, [1 / 3, 2 / 3])

    def assign_band(v):
        if v <= q1:
            return "low"
        if v <= q2:
            return "mid"
        return "high"

    sub["seed_band"] = [assign_band(v) for v in values]
    per_band = max(1, seed_count // 3)

    picked = []
    for band in SEED_BANDS:
        band_df = sub[sub["seed_band"] == band].copy()
        if band_df.empty:
            continue
        sample_idx = np.linspace(0, len(band_df) - 1, min(per_band, len(band_df)), dtype=int)
        picked.append(band_df.iloc[sample_idx].copy())

    seeds = pd.concat(picked, ignore_index=True) if picked else sub.head(0).copy()

    if len(seeds) < seed_count:
        existing = set(seeds["canonical_smiles"].tolist())
        remainder = sub[~sub["canonical_smiles"].isin(existing)].copy()
        if len(remainder):
            extra_idx = np.linspace(0, len(remainder) - 1, min(seed_count - len(seeds), len(remainder)), dtype=int)
            seeds = pd.concat([seeds, remainder.iloc[extra_idx].copy()], ignore_index=True)

    seeds = seeds.head(seed_count).reset_index(drop=True)
    seeds["seed_index"] = np.arange(len(seeds), dtype=int)
    seeds["seed_raw_property"] = pd.to_numeric(seeds[property_name], errors="coerce")
    return seeds

## 8. Run the corrected single-property experiment

In [ ]:
seed_df = select_property_seeds(val_panel, PROPERTY, SEED_COUNT)
assert not seed_df.empty, f"No valid validation seeds found for {PROPERTY}."

seed_registry_csv = TABLES_DIR / f"{PROPERTY.lower()}_seed_registry.csv"
seed_df.to_csv(seed_registry_csv, index=False)

print("Saved:", seed_registry_csv)
display(seed_df[["seed_index", "seed_band", "dataset_index", PROPERTY, "canonical_smiles"]].head(12))

all_candidate_rows = []
all_medoid_rows = []
all_path_rows = []
all_seed_summaries = []
all_step_summaries = []

for seed_row in tqdm(seed_df.to_dict("records"), desc=f"{PROPERTY} seeds"):
    dataset_index = int(seed_row["dataset_index"])
    seed_tensor = torch.as_tensor(padded_data[dataset_index], dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        seed_mu, _ = model.encode(seed_tensor)

    seed_mu_np = seed_mu[0].detach().cpu().numpy().astype(np.float32)
    z0_std = raw_to_std_latents(seed_mu_np)

    seed_decode_free = decode_latents(seed_mu, forced_len=None)
    seed_pred_len = int(seed_decode_free["pred_len_used"].iloc[0])

    seed_fp = make_morgan_fp(smiles_to_mol(seed_row["canonical_smiles"]))

    for spec_row in direction_specs:
        variant = spec_row["direction_variant"]
        span_label = spec_row["span_label"]
        direction_std = np.asarray(spec_row["direction_unit_std"], dtype=np.float32)
        step_value_std_values = np.asarray(spec_row["step_values_std"], dtype=float)
        z_centers_std = z0_std[None, :] + step_value_std_values[:, None] * direction_std[None, :]

        for decode_mode in DECODE_MODES:
            forced_len = None if decode_mode == "free_length" else seed_pred_len
            sigma_std_current = SIGMA_STD_LOW
            early_uniqueness = []
            step_candidate_groups = []
            candidate_rows_local = []
            medoid_rows_local = []

            for step_index, (step_signed, step_value_std) in enumerate(zip(STEP_GRID.astype(float), step_value_std_values)):
                z_center_std = z_centers_std[step_index]
                extra_count = max(int(PACK_SIZE) - 1, 0)

                noise_seed = (
                    SEED
                    + int(seed_row["seed_index"]) * 100000
                    + int(step_index) * 100
                    + stable_random_index(spec_row["direction_label"], 100000)
                )
                noise_std = orthogonal_noise_std(direction_std, sigma_std_current, extra_count, noise_seed)

                if extra_count > 0:
                    z_pack_std = np.concatenate([
                        z_center_std[None, :],
                        z_center_std[None, :] + noise_std,
                    ], axis=0)
                else:
                    z_pack_std = z_center_std[None, :]

                z_pack_raw = std_to_raw_latents(z_pack_std)
                decoded_pack = decode_latents(z_pack_raw, forced_len=forced_len)
                decoded_pack["candidate_index"] = np.arange(len(decoded_pack), dtype=int)
                decoded_pack["property"] = PROPERTY
                decoded_pack["direction_variant"] = variant
                decoded_pack["direction_label"] = spec_row["direction_label"]
                decoded_pack["span_multiplier"] = float(spec_row["span_multiplier"])
                decoded_pack["span_label"] = span_label
                decoded_pack["direction_source"] = spec_row["direction_source"]
                decoded_pack["direction_space"] = spec_row["direction_space"]
                decoded_pack["decode_mode"] = decode_mode
                decoded_pack["seed_index"] = int(seed_row["seed_index"])
                decoded_pack["seed_band"] = seed_row["seed_band"]
                decoded_pack["seed_dataset_index"] = dataset_index
                decoded_pack["seed_canonical_smiles"] = seed_row["canonical_smiles"]
                decoded_pack["seed_raw_property"] = float(seed_row["seed_raw_property"])
                decoded_pack["step_index"] = int(step_index)
                decoded_pack["step_signed"] = float(step_signed)
                decoded_pack["step_value_std"] = float(step_value_std)
                decoded_pack["target_metric_name"] = spec_row["target_name"]
                decoded_pack["confound_name"] = spec_row["confound_name"]
                decoded_pack["delta_prop_nominal"] = float(spec_row["delta_prop_nominal"]) if np.isfinite(spec_row["delta_prop_nominal"]) else np.nan
                decoded_pack["delta_step_std_nominal"] = float(spec_row["delta_step_std_nominal"]) if np.isfinite(spec_row["delta_step_std_nominal"]) else np.nan
                decoded_pack["delta_step_std_effective"] = float(spec_row["delta_step_std_effective"])
                decoded_pack["target_total_span_std"] = float(spec_row["target_total_span_std"])
                decoded_pack["benchmark_total_span_std_ref"] = float(spec_row["benchmark_total_span_std_ref"])
                decoded_pack["predicted_delta_prop_effective"] = (
                    float(step_value_std) * float(spec_row["w_std_norm"])
                    if variant != "random" and np.isfinite(spec_row["w_std_norm"])
                    else np.nan
                )
                decoded_pack["sigma_std_used"] = float(sigma_std_current)

                feature_df = build_candidate_features(decoded_pack, seed_fp)
                decoded_pack = pd.concat([decoded_pack.reset_index(drop=True), feature_df.reset_index(drop=True)], axis=1)

                medoid_row = within_step_medoid(decoded_pack)
                medoid_row["sigma_std_used"] = float(sigma_std_current)
                medoid_rows_local.append(medoid_row.to_dict())

                candidate_rows_local.extend(decoded_pack.to_dict("records"))
                step_candidate_groups.append(decoded_pack)

                valid_count = int((decoded_pack["valid"] == 1).sum())
                unique_valid = int(decoded_pack.loc[decoded_pack["valid"] == 1, "canonical_smiles"].nunique())
                early_uniqueness.append(unique_valid / max(valid_count, 1) if valid_count > 0 else 0.0)

                if step_index == min(2, len(step_value_std_values) - 1):
                    if float(np.mean(early_uniqueness)) < EARLY_UNIQUENESS_THRESHOLD:
                        sigma_std_current = SIGMA_STD_HIGH

            dp_indices, dp_total_cost = dynamic_programming_path(step_candidate_groups)
            greedy_indices, greedy_total_cost = greedy_medoid_path_cost(step_candidate_groups)

            path_rows_local = []
            for step_pos, (step_df, dp_idx) in enumerate(zip(step_candidate_groups, dp_indices)):
                chosen = step_df[step_df["candidate_index"] == dp_idx].iloc[0].copy()
                chosen["representative_strategy"] = "dynamic_programming"
                chosen["dp_path_total_cost"] = float(dp_total_cost)
                chosen["greedy_medoid_total_cost"] = float(greedy_total_cost)
                chosen["greedy_medoid_candidate_index"] = int(greedy_indices[step_pos])
                path_rows_local.append(chosen.to_dict())

            path_df_local = pd.DataFrame(path_rows_local)
            path_df_local = add_path_level_metrics(path_df_local, seed_row)

            seed_summary, step_summary = summarize_seed_run(
                pd.DataFrame(candidate_rows_local),
                path_df_local,
                seed_row,
                spec_row,
            )
            seed_summary["property"] = PROPERTY
            seed_summary["direction_variant"] = variant
            seed_summary["span_label"] = span_label
            seed_summary["span_multiplier"] = float(spec_row["span_multiplier"])
            seed_summary["decode_mode"] = decode_mode
            step_summary["property"] = PROPERTY
            step_summary["direction_variant"] = variant
            step_summary["span_label"] = span_label
            step_summary["span_multiplier"] = float(spec_row["span_multiplier"])
            step_summary["decode_mode"] = decode_mode
            step_summary["seed_index"] = int(seed_row["seed_index"])

            all_candidate_rows.extend(candidate_rows_local)
            all_medoid_rows.extend(medoid_rows_local)
            all_path_rows.extend(path_df_local.to_dict("records"))
            all_seed_summaries.append(seed_summary)
            all_step_summaries.extend(step_summary.to_dict("records"))

candidate_df = pd.DataFrame(all_candidate_rows)
medoid_df = pd.DataFrame(all_medoid_rows)
path_df = pd.DataFrame(all_path_rows)
seed_summary_df = pd.DataFrame(all_seed_summaries)
step_summary_df = pd.DataFrame(all_step_summaries)

assert not candidate_df.empty, "No candidate decodes were produced."
assert not path_df.empty, "No representative paths were produced."

candidate_csv = TABLES_DIR / f"{PROPERTY.lower()}_candidate_level.csv"
medoid_csv = TABLES_DIR / f"{PROPERTY.lower()}_medoid_level.csv"
path_csv = TABLES_DIR / f"{PROPERTY.lower()}_representative_paths.csv"
seed_summary_csv = TABLES_DIR / f"{PROPERTY.lower()}_seed_summary.csv"
step_summary_csv = TABLES_DIR / f"{PROPERTY.lower()}_step_summary.csv"

candidate_df.drop(columns=["candidate_fp"], errors="ignore").to_csv(candidate_csv, index=False)
medoid_df.drop(columns=["candidate_fp"], errors="ignore").to_csv(medoid_csv, index=False)
path_df.drop(columns=["candidate_fp"], errors="ignore").to_csv(path_csv, index=False)
seed_summary_df.to_csv(seed_summary_csv, index=False)
step_summary_df.to_csv(step_summary_csv, index=False)

print("Saved:", candidate_csv)
print("Saved:", medoid_csv)
print("Saved:", path_csv)
print("Saved:", seed_summary_csv)
print("Saved:", step_summary_csv)
print("candidate_rows:", len(candidate_df))
print("path_rows:", len(path_df))
display(seed_summary_df.head(12))

## 9. Aggregate plots, representative strips, validation checks, and run summary

In [ ]:
run_summary_df = (
    seed_summary_df
    .groupby(["property", "direction_variant", "span_label", "decode_mode"], as_index=False)
    .agg(
        span_multiplier=("span_multiplier", "first"),
        seeds=("seed_index", "nunique"),
        target_spearman_median=("target_spearman", "median"),
        target_slope_median=("target_slope", "median"),
        monotonic_violations_median=("target_monotonic_violation_count", "median"),
        validity_fraction_mean=("validity_fraction", "mean"),
        uniqueness_fraction_mean=("uniqueness_fraction_valid", "mean"),
        path_unique_canonical_median=("path_unique_canonical_count", "median"),
        path_constant_fraction_mean=("path_constant_across_steps", "mean"),
        path_similarity_end_median=("path_similarity_to_seed_end", "median"),
        adjacent_tanimoto_median=("adjacent_step_tanimoto_median", "median"),
        confound_slope_median=("confound_slope", "median"),
        pred_len_slope_median=("pred_len_slope", "median"),
        dp_no_worse_fraction=("dp_beats_or_matches_medoid", "mean"),
    )
)
run_summary_csv = TABLES_DIR / f"{PROPERTY.lower()}_run_summary.csv"
run_summary_df.to_csv(run_summary_csv, index=False)
print("Saved:", run_summary_csv)
display(run_summary_df)

def plot_curve_panel():
    variants = ["raw", "residual", "confound", "random"]
    span_labels = [format_span_label(x) for x in SPAN_SWEEP_MULTIPLIERS]
    n_rows = len(DECODE_MODES) * len(span_labels)
    fig, axes = plt.subplots(n_rows, 4, figsize=(24, 4.8 * n_rows), sharex=False)
    axes = np.asarray(axes)
    if axes.ndim == 1:
        axes = axes[None, :]

    for decode_idx, decode_mode in enumerate(DECODE_MODES):
        mode_path = path_df[path_df["decode_mode"] == decode_mode].copy()
        for span_idx, span_label in enumerate(span_labels):
            row_idx = decode_idx * len(span_labels) + span_idx
            sweep_path = mode_path[mode_path["span_label"] == span_label].copy()

            target_ax = axes[row_idx, 0]
            resid_ax = axes[row_idx, 1]
            conf_ax = axes[row_idx, 2]
            len_ax = axes[row_idx, 3]

            for variant in variants:
                variant_path = sweep_path[sweep_path["direction_variant"] == variant].copy()
                if variant_path.empty:
                    continue
                plot_df = variant_path.groupby("step_value_std", as_index=False).agg(
                    target_median=(PROPERTY, "median"),
                    residual_median=(PROPERTY_RESID_NAME, "median"),
                    confound_median=(linked_confound, "median"),
                    pred_len_median=("pred_len_used", "median"),
                )
                target_ax.plot(plot_df["step_value_std"], plot_df["target_median"], marker="o", label=variant)
                resid_ax.plot(plot_df["step_value_std"], plot_df["residual_median"], marker="o", label=variant)
                conf_ax.plot(plot_df["step_value_std"], plot_df["confound_median"], marker="o", label=variant)
                len_ax.plot(plot_df["step_value_std"], plot_df["pred_len_median"], marker="o", label=variant)

            target_ax.set_title(f"{PROPERTY} median ({decode_mode}, {span_label})")
            resid_ax.set_title(f"{PROPERTY_RESID_NAME} median ({decode_mode}, {span_label})")
            conf_ax.set_title(f"{linked_confound} median ({decode_mode}, {span_label})")
            len_ax.set_title(f"Predicted length median ({decode_mode}, {span_label})")

            for ax in axes[row_idx]:
                ax.set_xlabel("step value (standardized latent)")
                ax.legend(fontsize=8)

    fig.tight_layout()
    out_path = PLOTS_DIR / f"{PROPERTY.lower()}_curves.png"
    fig.savefig(out_path, dpi=180)
    plt.close(fig)
    return out_path

def plot_quality_panel():
    variants = ["raw", "residual", "confound", "random"]
    span_labels = [format_span_label(x) for x in SPAN_SWEEP_MULTIPLIERS]
    n_rows = len(DECODE_MODES) * len(span_labels)
    fig, axes = plt.subplots(n_rows, 3, figsize=(20, 4.5 * n_rows), sharex=False)
    axes = np.asarray(axes)
    if axes.ndim == 1:
        axes = axes[None, :]

    for decode_idx, decode_mode in enumerate(DECODE_MODES):
        mode_steps = step_summary_df[step_summary_df["decode_mode"] == decode_mode].copy()
        mode_path = path_df[path_df["decode_mode"] == decode_mode].copy()
        for span_idx, span_label in enumerate(span_labels):
            row_idx = decode_idx * len(span_labels) + span_idx
            sweep_steps = mode_steps[mode_steps["span_label"] == span_label].copy()
            sweep_path = mode_path[mode_path["span_label"] == span_label].copy()

            quality_ax = axes[row_idx, 0]
            similarity_ax = axes[row_idx, 1]
            continuity_ax = axes[row_idx, 2]

            for variant in variants:
                variant_steps = sweep_steps[sweep_steps["direction_variant"] == variant].copy()
                if not variant_steps.empty:
                    qdf = variant_steps.groupby("step_index", as_index=False).agg(
                        validity_fraction=("validity_fraction", "mean"),
                        uniqueness_fraction=("uniqueness_fraction", "mean"),
                    )
                    quality_ax.plot(qdf["step_index"], qdf["validity_fraction"], marker="o", label=f"{variant} valid")
                    quality_ax.plot(qdf["step_index"], qdf["uniqueness_fraction"], marker="x", linestyle="--", label=f"{variant} uniq")

                variant_path = sweep_path[sweep_path["direction_variant"] == variant].copy()
                if not variant_path.empty:
                    sdf = variant_path.groupby("step_value_std", as_index=False).agg(
                        seed_similarity_median=("seed_similarity", "median"),
                        adjacent_tanimoto_median=("adjacent_step_tanimoto", "median"),
                    )
                    similarity_ax.plot(sdf["step_value_std"], sdf["seed_similarity_median"], marker="o", label=variant)
                    continuity_ax.plot(sdf["step_value_std"], sdf["adjacent_tanimoto_median"], marker="o", label=variant)

            quality_ax.set_title(f"Validity / uniqueness ({decode_mode}, {span_label})")
            similarity_ax.set_title(f"Seed similarity ({decode_mode}, {span_label})")
            continuity_ax.set_title(f"Adjacent-step Tanimoto ({decode_mode}, {span_label})")

            quality_ax.set_xlabel("step index")
            similarity_ax.set_xlabel("step value (standardized latent)")
            continuity_ax.set_xlabel("step value (standardized latent)")
            quality_ax.set_ylim(0.0, 1.05)
            similarity_ax.set_ylim(0.0, 1.05)
            continuity_ax.set_ylim(0.0, 1.05)

            quality_ax.legend(fontsize=7, ncol=2)
            similarity_ax.legend(fontsize=8)
            continuity_ax.legend(fontsize=8)

    fig.tight_layout()
    out_path = PLOTS_DIR / f"{PROPERTY.lower()}_quality.png"
    fig.savefig(out_path, dpi=180)
    plt.close(fig)
    return out_path

curve_plot_path = plot_curve_panel()
quality_plot_path = plot_quality_panel()

def save_strip_image(sub_df, out_path):
    if sub_df.empty:
        return
    sub_df = sub_df.sort_values("step_index").reset_index(drop=True)
    mols = [smiles_to_mol(s) for s in sub_df["smiles"].tolist()]
    legends = []
    for row in sub_df.itertuples(index=False):
        prop_str = f"{float(getattr(row, PROPERTY)):.2f}" if np.isfinite(getattr(row, PROPERTY)) else "nan"
        resid_str = f"{float(getattr(row, PROPERTY_RESID_NAME)):.2f}" if np.isfinite(getattr(row, PROPERTY_RESID_NAME)) else "nan"
        sim_str = f"{float(row.seed_similarity):.2f}" if np.isfinite(row.seed_similarity) else "nan"
        legends.append(f"step={int(row.step_signed)}\n{PROPERTY}={prop_str}\nresid={resid_str}\nsim={sim_str}")
    img = Draw.MolsToGridImage(
        mols,
        molsPerRow=len(mols),
        subImgSize=(260, 260),
        legends=legends,
        useSVG=False,
        returnPNG=True,
    )
    if isinstance(img, (bytes, bytearray)):
        out_path.write_bytes(bytes(img))
    elif hasattr(img, "save"):
        img.save(str(out_path))
    elif hasattr(img, "data") and isinstance(img.data, (bytes, bytearray)):
        out_path.write_bytes(bytes(img.data))
    else:
        raise TypeError(f"Unsupported RDKit image type: {type(img)!r}")

strip_rows = []
for variant in REPRESENTATIVE_VARIANTS_TO_STRIP:
    variant_paths = path_df[
        (path_df["direction_variant"] == variant)
        & (path_df["decode_mode"] == STRIP_DECODE_MODE)
        & (path_df["span_label"] == STRIP_SPAN_LABEL)
    ].copy()
    if variant_paths.empty:
        continue
    for band in SEED_BANDS:
        band_seed_rows = seed_df[seed_df["seed_band"] == band].sort_values(PROPERTY).head(1)
        if band_seed_rows.empty:
            continue
        seed_index = int(band_seed_rows["seed_index"].iloc[0])
        strip_df = variant_paths[variant_paths["seed_index"] == seed_index].copy()
        if strip_df.empty:
            continue
        out_path = STRIPS_DIR / f"{PROPERTY.lower()}_{variant}_{band}_{STRIP_SPAN_LABEL}_{STRIP_DECODE_MODE}.png"
        save_strip_image(strip_df, out_path)
        strip_rows.append({
            "property": PROPERTY,
            "direction_variant": variant,
            "seed_band": band,
            "span_label": STRIP_SPAN_LABEL,
            "seed_index": seed_index,
            "decode_mode": STRIP_DECODE_MODE,
            "path": str(out_path),
        })

strip_index_df = pd.DataFrame(strip_rows)
strip_index_csv = TABLES_DIR / f"{PROPERTY.lower()}_strip_index.csv"
strip_index_df.to_csv(strip_index_csv, index=False)

expected_step_count = len(STEP_GRID)
expected_candidate_rows_per_run = expected_step_count * PACK_SIZE

validation_rows = []
for spec_row in direction_specs:
    variant = spec_row["direction_variant"]
    span_label = spec_row["span_label"]
    for decode_mode in DECODE_MODES:
        run_candidates = candidate_df[
            (candidate_df["direction_variant"] == variant)
            & (candidate_df["span_label"] == span_label)
            & (candidate_df["decode_mode"] == decode_mode)
        ]
        run_paths = path_df[
            (path_df["direction_variant"] == variant)
            & (path_df["span_label"] == span_label)
            & (path_df["decode_mode"] == decode_mode)
        ]

        if run_candidates.empty and run_paths.empty:
            continue

        candidate_rows_ok = bool((run_candidates.groupby("seed_index").size() == expected_candidate_rows_per_run).all())
        path_rows_ok = bool((run_paths.groupby("seed_index").size() == expected_step_count).all())

        fixed_length_ok = True
        if decode_mode == "fixed_length_seed_pred" and not run_candidates.empty:
            fixed_len_counts = run_candidates.groupby("seed_index")["forced_len"].nunique(dropna=True)
            fixed_length_ok = bool((fixed_len_counts <= 1).all())

        geometry_max_abs_error = np.nan
        if variant in {"raw", "residual", "confound"}:
            check_values = []
            for seed_row in seed_df.head(min(2, len(seed_df))).to_dict("records"):
                z0_raw = z_raw[int(seed_row["dataset_index"])]
                z0_std = raw_to_std_latents(z0_raw)
                for step_value_std in np.asarray(spec_row["step_values_std"], dtype=float):
                    zt_raw = std_to_raw_latents(z0_std + step_value_std * np.asarray(spec_row["direction_unit_std"], dtype=np.float32))
                    predicted_delta = float(np.dot(np.asarray(spec_row["w_raw"], dtype=np.float32), zt_raw - z0_raw))
                    intended_delta = float(step_value_std * spec_row["w_std_norm"])
                    check_values.append(abs(predicted_delta - intended_delta))
            geometry_max_abs_error = float(max(check_values)) if check_values else np.nan

        dp_no_worse = bool(
            (
                seed_summary_df[
                    (seed_summary_df["direction_variant"] == variant)
                    & (seed_summary_df["span_label"] == span_label)
                    & (seed_summary_df["decode_mode"] == decode_mode)
                ]["dp_beats_or_matches_medoid"] >= 1.0
            ).all()
        )

        validation_rows.append({
            "property": PROPERTY,
            "direction_variant": variant,
            "span_label": span_label,
            "span_multiplier": float(spec_row["span_multiplier"]),
            "decode_mode": decode_mode,
            "expected_seeds": int(seed_df["seed_index"].nunique()),
            "candidate_rows_ok": candidate_rows_ok,
            "path_rows_ok": path_rows_ok,
            "fixed_length_ok": fixed_length_ok,
            "geometry_max_abs_error": geometry_max_abs_error,
            "dp_no_worse_than_medoid": dp_no_worse,
            "benchmark_total_span_std_ref": float(benchmark_total_span_std_ref),
            "default_total_span_std": float(DEFAULT_TOTAL_SPAN_STD),
            "max_total_span_std": float(MAX_TOTAL_SPAN_STD),
        })

validation_df = pd.DataFrame(validation_rows)
validation_csv = TABLES_DIR / f"{PROPERTY.lower()}_validation_checks.csv"
validation_df.to_csv(validation_csv, index=False)

issues = []
if not validation_df["candidate_rows_ok"].all():
    issues.append("candidate_rows_ok")
if not validation_df["path_rows_ok"].all():
    issues.append("path_rows_ok")
if not validation_df["fixed_length_ok"].all():
    issues.append("fixed_length_ok")
if not validation_df["dp_no_worse_than_medoid"].all():
    issues.append("dp_no_worse_than_medoid")
if validation_df["geometry_max_abs_error"].notna().any():
    if float(validation_df["geometry_max_abs_error"].dropna().max()) >= 2e-4:
        issues.append("geometry_max_abs_error")

summary = {
    "property": PROPERTY,
    "property_residual_name": PROPERTY_RESID_NAME,
    "linked_confound": linked_confound,
    "smoke_mode": bool(SMOKE_MODE),
    "seed_count": int(SEED_COUNT),
    "pack_size": int(PACK_SIZE),
    "decode_modes": list(DECODE_MODES),
    "step_grid": STEP_GRID.astype(int).tolist(),
    "benchmark_total_span_std_ref": float(benchmark_total_span_std_ref),
    "default_total_span_std": float(DEFAULT_TOTAL_SPAN_STD),
    "max_total_span_std": float(MAX_TOTAL_SPAN_STD),
    "span_sweep_multipliers": [float(x) for x in SPAN_SWEEP_MULTIPLIERS],
    "span_sweep_labels": list(SPAN_SWEEP_LABELS),
    "strip_span_label": STRIP_SPAN_LABEL,
    "sweep_max_total_span_std": float(SWEEP_MAX_TOTAL_SPAN_STD),
    "step4_available": bool(STEP4_AVAILABLE),
    "model_py": str(MODEL_PY),
    "ckpt_path": str(CKPT_PATH),
    "probe_summary_csv": str(probe_summary_csv),
    "alignment_csv": str(alignment_csv),
    "residual_alignment_csv": str(resid_alignment_csv),
    "direction_specs_csv": str(direction_specs_csv),
    "seed_registry_csv": str(seed_registry_csv),
    "candidate_csv": str(candidate_csv),
    "medoid_csv": str(medoid_csv),
    "path_csv": str(path_csv),
    "seed_summary_csv": str(seed_summary_csv),
    "step_summary_csv": str(step_summary_csv),
    "run_summary_csv": str(run_summary_csv),
    "validation_csv": str(validation_csv),
    "curve_plot_path": str(curve_plot_path),
    "quality_plot_path": str(quality_plot_path),
    "strip_index_csv": str(strip_index_csv),
    "validation_issues": issues,
}
summary_json = OUTPUT_DIR / f"summary_{PROPERTY.lower()}_single_property_direction_validation_span_sweep.json"
summary_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Saved:", curve_plot_path)
print("Saved:", quality_plot_path)
print("Saved:", strip_index_csv)
print("Saved:", validation_csv)
print("Saved:", summary_json)

if issues:
    print("VALIDATION WARNINGS:", issues)
else:
    print("Validation checks passed.")

display(validation_df)
print(json.dumps(summary, indent=2))